## 콘텐츠 데이터


### 네이버 블로그 크롤링


In [1]:
import datetime
import json
import re
import time
from urllib.parse import urlencode, urljoin, urlparse
from zoneinfo import ZoneInfo

import requests
from bs4 import BeautifulSoup

In [2]:
# 검색 조건 설정
query = "삼성전자"
max_results = 100
sort = {"relevance": 0, "latest": 1}["relevance"]
sort_option = "dd" if sort == 1 else "r"
timeout = 10
delay = 0.7

# 네이버 블로그 검색 주소
search_url = "https://search.naver.com/search.naver"

# 일반 웹 브러우저 요청처럼 보이도록 설정
header = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
    ),
}

# 여러 요청에서 동일한 헤더를 사용
session = requests.Session()
session.headers.update(header)

In [3]:
# 검색 결과에서 수집한 블로그 URL과 중복 확인용 집합
blog_urls = []
seen_posts = set()
page = 0

# 원하는 개수만큼 검색 결과 페이지를 순회
while len(blog_urls) < max_results:
    search_params = {
        "where": "blog",
        "ssc": "tab.blog.all",
        "sm": "tab_jum",
        "query": query,
        "nso": f"so:{sort_option},p:all,a:all",
        "start": page * 30 + 1,
    }

    response = session.get(
        search_url,
        params=search_params,
        timeout=timeout,
    )
    response.raise_for_status()

    search_soup = BeautifulSoup(response.text, "html.parser")
    found_this_page = 0

    # 검색 결과의 모든 링크 확인
    for link in search_soup.select("a[href]"):
        href = link.get("href") or ""
        parsed = urlparse(href)

        # 네이버 블로그 링크만 처리
        if parsed.netloc not in {"blog.naver.com"}:
            continue

        # URL에서 블로그 ID와 게시글 ID 추출
        match = re.match(r"^/([^/?#]+)/(\d+)(?:/|$)", parsed.path)
        if not match:
            continue

        blog_id, post_id = match.groups()

        post_url = f"https://blog.naver.com/{blog_id}/{post_id}"

        # 이미 수집한 게시글은 제외
        if post_url in seen_posts:
            continue

        seen_posts.add(post_url)
        blog_urls.append(post_url)
        found_this_page += 1

        if len(blog_urls) >= max_results:
            break

    page += 1

    # 다음 페이지 요청 전 잠시 대기
    time.sleep(delay)

print(f"수집할 블로그 URL: {len(blog_urls)}개")
blog_urls[:3]

수집할 블로그 URL: 100개


['https://blog.naver.com/bomod/224356221110',
 'https://blog.naver.com/jogyo_j/224361645264',
 'https://blog.naver.com/press02/224352115150']

In [4]:
# 수집한 블로그 게시글을 저장할 리스트
blogs = []

# 블로그 게시글 URL을 하나씩 순회
for index, post_url in enumerate(blog_urls):
    try:
        # URL에서 블로그 ID와 게시글 ID 추출
        match = re.match(r"^https://blog\.naver\.com/([^/]+)/(\d+)", post_url)
        if match is None:
            raise ValueError("올바른 네이버 블로그 게시글 URL이 아닙니다.")

        blog_id, post_id = match.groups()

        # 게시글 본문을 직접 조회할 PostView URL 생성
        postview_url = "https://blog.naver.com/PostView.naver?" + urlencode(
            {"blogId": blog_id, "logNo": post_id}
        )

        response = session.get(postview_url, timeout=timeout)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # 게시글 제목 추출
        title_element = soup.select_one(".pcol1")
        if title_element is None:
            raise RuntimeError("기사 제목 영역을 찾지 못했습니다.")

        title = " ".join(title_element.get_text(" ", strip=True).split())

        # 게시글 본문 영역 추출
        content = soup.select_one(".se-main-container")
        if content is None:
            raise RuntimeError("게시글 본문 영역을 찾지 못했습니다.")

        # 본문 문단을 정리하여 저장
        content_lines = []
        for paragraph in content.select(".se-text-paragraph"):
            text = paragraph.get_text(" ", strip=True)
            text = text.replace("\u200b", "")
            text = " ".join(text.split())

            if text:
                content_lines.append(text)

        content_text = "\n".join(content_lines)

        # 본문에 포함된 이미지와 설명 저장
        images = []
        seen_image_urls = set()

        for image_component in content.select(".se-component.se-image"):
            image = image_component.select_one("img.se-image-resource")
            if image is None:
                continue

            image_url = (image.get("src") or "").strip()
            if not image_url:
                continue

            image_url = urljoin(postview_url, image_url)
            if image_url in seen_image_urls:
                continue

            seen_image_urls.add(image_url)

            # 이미지 설명 추출
            caption_element = image_component.select_one(".se-caption")
            caption = ""

            if caption_element:
                caption = caption_element.get_text(" ", strip=True)
                caption = " ".join(caption.split())

            images.append(
                {
                    "url": image_url,
                    "caption": caption,
                }
            )

        # 게시글 작성 시간 추출
        written_at_element = soup.select_one(".se_publishDate")
        if written_at_element is None:
            raise RuntimeError("게시글 작성 시간 영역을 찾지 못했습니다.")

        written_at_text = written_at_element.get_text(" ", strip=True)
        written_at_text = " ".join(written_at_text.split())

        written_at = (
            datetime.datetime.strptime(
                written_at_text,
                "%Y. %m. %d. %H:%M",
            )
            .replace(tzinfo=ZoneInfo("Asia/Seoul"))
            .isoformat()
        )

        # 수집한 게시글 정보를 리스트에 추가
        blogs.append(
            {
                "query": query,
                "title": title,
                "content": content_text,
                "images": images,
                "written_at": written_at,
                "url": post_url,
            }
        )

    # 네트워크 요청 관련 오류 처리
    except requests.RequestException as error:
        print(
            f"[{index + 1}/{len(blog_urls)}] 요청 실패: "
            f"{post_url} - {type(error).__name__}: {error}"
        )

    # URL 형식이나 본문 추출 관련 오류 처리
    except (ValueError, RuntimeError) as error:
        print(f"[{index + 1}/{len(blog_urls)}] 수집 실패: {post_url} - {error}")

print(f"본문 수집 성공: {len(blogs)}개")


본문 수집 성공: 100개


In [5]:
# 첫 번째 수집 결과 확인
blogs[0]

{'query': '삼성전자',
 'title': '27만원까지 떨어진 삼성전자 주가 증권사 목표가는 무려...',
 'content': '우리 삼성전자, 이제 60만 전자 가는 걸까요?\n지금 27만원인데...\n요즘 삼성전자뿐 아니라 반도체주 들고 계신 분들은 하루 오르면 다음 날 빠지고, 또 오르면 다시 밀리고...\n솔직히 언제쯤 다시 제대로 올라갈지 감도 안 잡히셨을 겁니다.\n그런데 구글 실적이 발표된 다음 날, 삼성전자를 비롯한 반도체주들이 일제히 크게 반등하더라고요.\n처음엔 저도 \'구글이 미국 회사인데 삼성전자랑 무슨 상관이지?\' 싶었습니다.\n찾아보니 시장이 그동안은 \'AI 투자도 이제 너무 많이 했다. 반도체 수요도 곧 꺾이는 거 아니냐\' 는 걱정으로 반도체주를 계속 팔고 있었더라고요.\n그런데 정작 구글은 "우리는 AI 투자 더 한다. 반도체도 계속 필요하다." 는 신호를 시장에 던진 셈이었고요.\n그러자 증권사들도 분위기가 조금씩 바뀌기 시작했습니다.\n심지어 KB증권은 삼성전자 목표주가를 60만원으로 그대로 유지 했더라고요.\n도대체 구글 실적에서 뭘 봤길래 아직도 삼성전자를 그렇게 좋게 보는 건지, 저도 궁금해서 보고서를 하나씩 뜯어봤습니다.\n구글이 쓰는 돈, 삼성전자로 얼마나 흘러가나\n이번 분석의 출발점은 구글이 AI 인프라에 쏟아붓는 돈의 방향인데요.\n구글은 내년까지 수백억 달러를 투입해서 외부 데이터센터 사업자의 임대료까지 직접 보증하겠다는 계획을 밝혔습니다.\n데이터센터를 짓는 쪽이 수요 걱정 없이 설비를 늘릴 수 있도록, 구글이 먼저 자리를 깔아주는 셈이죠.\n올해 구글 한 곳의 AI 투자 규모만 미국 빅테크 전체 AI 투자액의 4분의 1을 넘길 것으로 추정되고요.\n여기서 삼성전자가 중요해지는 이유는, 구글 서버 메모리 공급에서 점유율 50%, HBM에서는 80%를 차지 하고 있다는 점 때문입니다.\n구글의 AI 투자가 늘어날수록, 그 예산의 상당 부분이 자연스럽게 삼성전자 매출로 이어지진다는 게 리포트가 말하

In [6]:
# 수집한 블로그를 JSONL 파일로 저장
output_path = "contents.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    file.writelines(json.dumps(blog, ensure_ascii=False) + "\n" for blog in blogs)

print(f"저장 완료: {output_path}")

저장 완료: contents.jsonl
